# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0–1 are one-time setup steps; Sections 2–4 are the main training and evaluation pipeline.

In [ ]:
# ── 全部步骤 ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, os
PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026"
if PROJECT_ROOT not in sys.path:
      sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

  # 安装依赖
os.system("pip install -r requirements.txt -q")
os.system("python -m spacy download en")

  # 下载数据（已有则跳过）
from Tools.download import download_mini
download_mini(data_dir="_data")

  # 预处理（已有则可跳过）
from Tools.preproc import preprocess
preprocess(
      train_file="_data/squad/train-mini.json",
      dev_file="_data/squad/dev-v1.1.json",
      glove_word_file="_data/glove/glove.mini.txt",
      target_dir="_data",
      para_limit=400,
      ques_limit=50,
  )

  # 训练
from TrainTools.train import train
results = train(
      train_npz       = "_data/train.npz",
      dev_npz         = "_data/dev.npz",
      word_emb_json   = "_data/word_emb.json",
      char_emb_json   = "_data/char_emb.json",
      train_eval_json = "_data/train_eval.json",
      dev_eval_json   = "_data/dev_eval.json",
      save_dir        = "_model",
      log_dir         = "_log",
      num_steps       = 60000,
      batch_size      = 8,
      seed            = 42,
      optimizer_name  = "adam",
      scheduler_name  = "lambda",
      loss_name       = "qa_nll",
      learning_rate   = 1e-3,
      beta1           = 0.8,
      beta2           = 0.999,
      eps             = 1e-7,
      weight_decay    = 3e-7,
      dropout         = 0.1,
      dropout_char    = 0.05,
      early_stop      = 15,
  )
print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path if your repo is stored elsewhere in Drive.
PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026"

In [ ]:
# Install Python dependencies (run once per session)
!pip install -r {PROJECT_ROOT}/requirements.txt -q
!python -m spacy download en

---
## Section 0 — Environment Setup

Mount Google Drive and install dependencies.

In [ ]:
import sys, os

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

---
## Section 1 — Download Data *(delete before submitting)*

Downloads the pre-built mini dataset (sampled SQuAD v1.1 train + full dev set,
with GloVe vectors filtered to the mini vocabulary) from GitHub Releases into `_data/`.

> **One-time step.** Once `_data/` exists on your Drive, delete this section before submission.

In [ ]:
from Tools.download import download_mini

download_mini(data_dir="_data")

---
## Section 2 — Preprocess Data *(delete before submitting)*

Tokenises the SQuAD JSON files, builds word/char vocabularies from GloVe, and writes padded index tensors to `_data/`.

> **One-time step.** Once `_data/*.npz` exists on your Drive, delete this section before submission. Re-run only if you change `para_limit`, `ques_limit`, or other shape parameters.

In [ ]:
from Tools.preproc import preprocess

preprocess(
    train_file="_data/squad/train-mini.json",
    dev_file="_data/squad/dev-v1.1.json",
    glove_word_file="_data/glove/glove.mini.txt",
    target_dir="_data",
    para_limit=400,
    ques_limit=50,
)

---
## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.

### Tuning Changelog

---

#### Run 1 — Baseline
**Config:** `optimizer=sgd`, `scheduler=none`, `lr=0.001`
**Result:** F1 = 4.2789, EM = 0.7500
**Problem:** SGD without momentum converges extremely slowly. No scheduler means constant lr with no warmup or decay. Model barely learned anything.

---

#### Run 2 — Adam + Lambda (Failed)
**Changes from Run 1:**

| # | Parameter | Before | After | Reason |
|---|-----------|--------|-------|--------|
| 1 | `optimizer_name` | `sgd` | `adam` | Adam has adaptive per-parameter learning rates suited for NLP/sparse gradients |
| 2 | `scheduler_name` | `none` | `lambda` | Intended to pair with Adam as per QANet paper design |
| 3 | `early_stop` | `10` | `15` | Give better optimizer more patience to converge |
| 4 | `beta1/beta2/eps/weight_decay` | defaults | `0.8/0.999/1e-7/3e-7` | Match QANet paper hyperparameters |

**Result:** F1 = 5.2248, EM = 1.5000, **Loss = ~10²⁶ ❌**
**Problem:** In this codebase, Adam's lr is hardcoded to `1.0`. The `lambda` scheduler multiplies by a constant `1.0`, so the effective learning rate is always `1.0` — far too large. This caused catastrophic gradient explosion (loss diverged to ~10²⁶).

---

#### Run 3 — SGDMomentum + Cosine (Current)
**Changes from Run 2:**

| # | Parameter | Before | After | Reason |
|---|-----------|--------|-------|--------|
| 1 | `optimizer_name` | `adam` | `sgd_momentum` | Adam's lr is hardcoded to 1.0 in this codebase, causing loss explosion. SGDMomentum uses `args.learning_rate` directly, giving full control over the lr. |
| 2 | `scheduler_name` | `lambda` | `cosine` | Cosine annealing smoothly decays lr from `learning_rate` → 0 over 60,000 steps, improving convergence and preventing oscillation near minima. |
| 3 | `learning_rate` | `1e-3` (SGD) / `1.0` (Adam internal) | `0.01` | A learning rate of 0.01 is standard for SGD with momentum on NLP tasks — high enough to learn quickly, low enough to stay stable. |
| 4 | `momentum` | N/A | `0.9` | Momentum accumulates gradient history, accelerating convergence in consistent directions and dampening oscillation. Standard value of 0.9. |

---
## Experiment 1 — Effect of Optimizer: SGD vs SGD with Momentum

**Research Question:** Does adding momentum to SGD improve QANet's convergence speed and final performance on SQuAD?

**Hypothesis:** SGD with momentum should converge faster and achieve better performance than vanilla SGD because momentum accumulates gradient history, accelerating updates in consistent directions and dampening oscillations.

**Setup:** All hyperparameters are fixed. Only `optimizer_name` changes between runs.

| Setting | Value |
|---------|-------|
| scheduler | cosine |
| learning_rate | 0.01 |
| weight_decay | 3e-7 |
| dropout | 0.1 |
| num_steps | 60000 |
| seed | 42 |

- **Exp 1a:** `optimizer_name = "sgd"` (no momentum)
- **Exp 1b:** `optimizer_name = "sgd_momentum"`, `momentum = 0.9` — results from Run 3

**Evaluation metrics:** Best F1, Best EM, training loss, convergence speed (steps to early stop).

In [ ]:
# Experiment 1a — SGD (no momentum)
# All settings identical to Run 3 (Exp 1b); only optimizer_name changes.
from TrainTools.train import train

results_exp1a = train(
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model_exp1a",
    log_dir         = "_log_exp1a",

    num_steps  = 60000,
    batch_size = 8,
    seed       = 42,

    scheduler_name = "cosine",
    loss_name      = "qa_nll",

    learning_rate = 0.01,
    weight_decay  = 3e-7,
    dropout       = 0.1,
    dropout_char  = 0.05,
    early_stop    = 15,

    # ── Only change: SGD without momentum ───────────────────────────────
    optimizer_name = "sgd",
)

print(f"[Exp 1a - SGD]  Best F1: {results_exp1a['best_f1']:.4f}  |  Best EM: {results_exp1a['best_em']:.4f}")

In [ ]:
from TrainTools.train import train

results = train(
    # ── data paths (must match preprocess outputs) ──────────────────────
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model",
    log_dir         = "_log",

    # ── training loop ────────────────────────────────────────────────────
    num_steps  = 60000,
    batch_size = 8,
    seed       = 42,

    # ── stable recipe: SGDMomentum + cosine scheduler ────────────────────
    optimizer_name = "sgd_momentum",
    scheduler_name = "cosine",
    loss_name      = "qa_nll",

    # ── optimizer hyperparameters ────────────────────────────────────────
    learning_rate = 0.01,
    momentum      = 0.9,
    weight_decay  = 3e-7,

    # ── regularization ───────────────────────────────────────────────────
    dropout       = 0.1,
    dropout_char  = 0.05,

    # ── early stopping ───────────────────────────────────────────────────
    early_stop    = 15,
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

---
## Experiment 2 — Effect of Normalization: Layer Norm vs Group Norm

**Research Question:** Does the choice of normalization method (Layer Norm vs Group Norm) affect QANet's training stability and final performance on SQuAD?

**Hypothesis:** Layer Norm is better suited for NLP tasks because it normalizes across the feature dimension independently for each token, while Group Norm may introduce inter-channel dependencies that are less appropriate for sequence models.

**Setup:** All hyperparameters are identical to the stable baseline (Run 3: SGDMomentum + Cosine). Only `norm_name` is changed between runs.

- **Exp 2a (Baseline):** `norm_name = "layer_norm"` — results from Run 3
- **Exp 2b (Treatment):** `norm_name = "group_norm"`, `norm_groups = 8`

**Evaluation metrics:** Best F1, Best EM, training loss curve, convergence speed.

In [ ]:
# Experiment 2b — Group Norm
# All settings identical to Run 3 baseline; only norm_name changes.
from TrainTools.train import train

results_exp2b = train(
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model_exp2b",
    log_dir         = "_log_exp2b",

    num_steps  = 60000,
    batch_size = 8,
    seed       = 42,

    optimizer_name = "sgd_momentum",
    scheduler_name = "cosine",
    loss_name      = "qa_nll",

    learning_rate = 0.01,
    momentum      = 0.9,
    weight_decay  = 3e-7,
    dropout       = 0.1,
    dropout_char  = 0.05,
    early_stop    = 15,

    # ── Only change: Group Norm ──────────────────────────────────────────
    norm_name   = "group_norm",
    norm_groups = 8,
)

print(f"[Exp 2b - GroupNorm]  Best F1: {results_exp2b['best_f1']:.4f}  |  Best EM: {results_exp2b['best_em']:.4f}")

---
## Experiment 3 — Effect of Weight Initialization: Kaiming vs Xavier

**Research Question:** Does the weight initialization strategy (Kaiming vs Xavier) affect QANet's convergence speed and final performance?

**Hypothesis:** Kaiming initialization is more suitable for networks with ReLU activations because it accounts for the fact that ReLU zeros out half the neurons (using a factor of 2/fan_in). Xavier initialization assumes linear activations and may lead to vanishing gradients in deeper ReLU networks.

**Setup:** All hyperparameters are identical to the stable baseline (Run 3). Only `init_name` is changed.

- **Exp 3a (Baseline):** `init_name = "kaiming"` — results from Run 3
- **Exp 3b (Treatment):** `init_name = "xavier"`

**Evaluation metrics:** Best F1, Best EM, early loss trajectory (convergence speed).

In [ ]:
# Experiment 3b — Xavier Initialization
# All settings identical to Run 3 baseline; only init_name changes.
from TrainTools.train import train

results_exp3b = train(
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model_exp3b",
    log_dir         = "_log_exp3b",

    num_steps  = 60000,
    batch_size = 8,
    seed       = 42,

    optimizer_name = "sgd_momentum",
    scheduler_name = "cosine",
    loss_name      = "qa_nll",

    learning_rate = 0.01,
    momentum      = 0.9,
    weight_decay  = 3e-7,
    dropout       = 0.1,
    dropout_char  = 0.05,
    early_stop    = 15,

    # ── Only change: Xavier initialization ──────────────────────────────
    init_name = "xavier",
)

print(f"[Exp 3b - Xavier]  Best F1: {results_exp3b['best_f1']:.4f}  |  Best EM: {results_exp3b['best_em']:.4f}")

---
## Section 4 — Evaluate

Loads the saved checkpoint and runs inference on the full dev set.

In [ ]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz       = "_data/dev.npz",
    word_emb_json = "_data/word_emb.json",
    char_emb_json = "_data/char_emb.json",
    dev_eval_json = "_data/dev_eval.json",
    save_dir      = "_model",
    log_dir       = "_log",
    ckpt_name     = "model.pt",
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")